# Training Loop, Loss, And Optimization

A transformer architecture is only useful if gradients, labels, and optimizer updates all line up. This notebook makes the training loop concrete before the package helpers compress it behind reusable APIs.


## Problem: What Must This Component Compute?

At training time, the model receives integer token tensors with shape `(B, T)`. The loop must build next-token targets, produce logits of shape `(B, T, V)`, compute cross-entropy, and update parameters without destabilizing gradients.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve optimization, scaling laws, or fine-tuning efficiency, it is usually changing one of these constraints: how examples are batched, how labels are aligned, how gradients are controlled, or how parameter updates are regularized.


## 1. Train/validation split.

Language-model training usually starts from one long token stream. A split keeps evaluation honest by holding out suffix tokens the optimizer never sees. That way the validation estimate measures generalization to unseen positions rather than replaying the exact data the optimizer used.


## 2. Label shifting.

Within each sampled block, the input and target differ by one position:

\[
x = [t_0, t_1, \ldots, t_{T-1}], \qquad
y = [t_1, t_2, \ldots, t_T].
\]

Every position predicts the next token rather than itself, which is why logits of shape `(B, T, V)` are compared against shifted targets of shape `(B, T)`.


In [ ]:
import torch

from llm_from_scratch.configs import ModelConfig, TrainConfig, get_device, profile_config, set_seed
from llm_from_scratch.data import get_batch, split_tokens
from llm_from_scratch.model import TinyGPT, count_parameters
from llm_from_scratch.train import overfit_tiny_batch, train_steps

set_seed(123)
runtime_device = get_device()
tokens = torch.arange(40, dtype=torch.long) % 16
train_tokens, val_tokens = split_tokens(tokens, train_fraction=0.8)
x, y = get_batch(train_tokens, block_size=6, batch_size=3, device=runtime_device)

assert x.shape == y.shape == (3, 6)
assert torch.equal(y[:, :-1], x[:, 1:])

{
    "runtime_device": runtime_device.type,
    "train_len": len(train_tokens),
    "val_len": len(val_tokens),
    "example_x": x[0].detach().cpu().tolist(),
    "example_y": y[0].detach().cpu().tolist(),
}


## 3. AdamW update rule.

AdamW keeps moving averages of gradients and squared gradients, rescales each parameter update adaptively, and then applies decoupled weight decay directly to parameter values. In words: Adam decides the update direction and scale, while weight decay separately nudges weights toward zero.


## 4. Gradient clipping.

Gradient clipping prevents unusually large updates from dominating a step. If the total norm exceeds a threshold `tau`, the gradient vector is rescaled so its norm becomes `tau` while its direction stays the same.


In [ ]:
param = torch.tensor([3.0, -4.0], requires_grad=True)
loss = (param ** 2).sum()
loss.backward()

pre_clip_norm = torch.linalg.vector_norm(param.grad).item()
total_norm = torch.nn.utils.clip_grad_norm_([param], max_norm=1.0)
post_clip_norm = torch.linalg.vector_norm(param.grad).item()

optimizer = torch.optim.AdamW([param], lr=1e-1, weight_decay=0.1)
before = param.detach().clone()
optimizer.step()
after = param.detach().clone()

assert pre_clip_norm > 1.0
assert post_clip_norm <= 1.00001
assert not torch.allclose(before, after)

{
    "reported_total_norm": float(total_norm),
    "pre_clip_norm": pre_clip_norm,
    "post_clip_norm": post_clip_norm,
    "parameter_delta": (after - before).tolist(),
}


## 5. Tiny overfit test.

A tiny overfit test is the fastest end-to-end training sanity check. If a small model cannot memorize a tiny batch, something is wrong with the loss, labels, gradients, optimizer, or module wiring. It is far cheaper to discover that on eight examples than after a long training run.


In [ ]:
import torch
from llm_from_scratch.configs import ModelConfig, set_seed
from llm_from_scratch.model import TinyGPT
from llm_from_scratch.train import overfit_tiny_batch

set_seed(123)
cfg = ModelConfig(vocab_size=16, block_size=8, n_embd=32, n_head=4, n_layer=1, dropout=0.0)
model = TinyGPT(cfg).to(runtime_device)
x = torch.randint(0, cfg.vocab_size, (8, cfg.block_size), device=runtime_device)
y = torch.randint(0, cfg.vocab_size, (8, cfg.block_size), device=runtime_device)
first_loss, last_loss = overfit_tiny_batch(model, x, y, steps=30, lr=1e-2)

assert last_loss < first_loss

{
    "runtime_device": runtime_device.type,
    "first_loss": first_loss,
    "last_loss": last_loss,
}


## 6. Quick versus study run configuration.

Quick runs are for feedback loops; study runs are for cleaner curves and better learning behavior. The curriculum package exposes both through `profile_config`, so runtime scale decisions stay explicit rather than getting scattered through notebook cells.


## 7. Loss curve interpretation.

Loss curves should be read qualitatively: a falling training curve shows the optimizer can fit the data, while a widening train/validation gap hints at overfitting or distribution mismatch. Flat or noisy curves usually mean a bug, an overly small model, or a poor hyperparameter choice.


In [ ]:
quick_model_cfg, quick_train_cfg = profile_config('quick')
study_model_cfg, study_train_cfg = profile_config('study')

set_seed(123)
tiny_cfg = ModelConfig(vocab_size=16, block_size=8, n_embd=32, n_head=4, n_layer=1, dropout=0.0)
tiny_model = TinyGPT(tiny_cfg)
train_stream = (torch.arange(256, dtype=torch.long) % tiny_cfg.vocab_size)
val_stream = (torch.arange(128, dtype=torch.long) % tiny_cfg.vocab_size)
history = train_steps(
    tiny_model,
    train_stream,
    val_stream,
    TrainConfig(
        batch_size=8,
        max_steps=12,
        eval_interval=6,
        eval_batches=2,
        learning_rate=1e-2,
        weight_decay=0.0,
        grad_clip=1.0,
        seed=123,
    ),
    runtime_device,
)

finite_history = [all(torch.isfinite(torch.tensor([row['train'], row['val']])).tolist()) for row in history]
assert len(history) == 2
assert all(finite_history)
assert study_train_cfg.max_steps > quick_train_cfg.max_steps

{
    "runtime_device": runtime_device.type,
    "quick_params": count_parameters(TinyGPT(quick_model_cfg)),
    "study_params": count_parameters(TinyGPT(study_model_cfg)),
    "quick_steps": quick_train_cfg.max_steps,
    "study_steps": study_train_cfg.max_steps,
    "history": history,
}


## Abstraction Bridge

The package helper `train_steps` wraps the same ingredients shown above: batch sampling, next-token loss, AdamW, optional gradient clipping, and periodic evaluation. `overfit_tiny_batch` isolates the minimum viable sanity check before longer profile-driven runs.


## Exercise Checkpoint 1

1. Why is the target tensor shifted left by one token relative to the input tensor?
2. If logits have shape `(B, T, V)`, which dimensions are being normalized inside cross-entropy?


## Exercise Checkpoint 2

1. What failure mode would you expect if gradient clipping is removed and training occasionally sees very large updates?
2. Why is a quick profile useful even when you already intend to do a longer study run later?
